# Preprocess

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import scanpy as sc
import pandas as pd
import numpy as np
import muon

In [ ]:
import os 
os.chdir('/path/to/hemelas_project/z_others/6_ann_visual/')

## RNA-seq preprocessing

In [ ]:
rna = sc.read_h5ad('./processed/adata_rna.h5ad')
rna

In [ ]:
rna.layers['counts'] = rna.X.copy()

In [ ]:
sc.pp.normalize_total(rna, target_sum=1e4)
sc.pp.log1p(rna)

In [ ]:
sc.pp.highly_variable_genes(rna, n_top_genes=4000, batch_key='sample')
rna

In [ ]:
rna_hvg = rna[:, rna.var.highly_variable].copy()
rna_hvg

In [ ]:
rna.write('../7_horizontal_RNA_ATAC/processed/rna.h5ad', compression='gzip')
rna_hvg.write('../7_horizontal_RNA_ATAC/processed/rna_hvg.h5ad', compression='gzip')

## ATAC-seq preprocessing

In [ ]:
atac = sc.read_h5ad('./processed/adata_atac.h5ad')
atac

### tfidf

In [ ]:
atac.layers['counts'] = atac.X.copy()

In [ ]:
muon.atac.pp.tfidf(atac, scale_factor=1e4)

In [ ]:
sc.pp.highly_variable_genes(atac, n_top_genes=60000, batch_key='sample')
atac

In [ ]:
atac_hvf = atac[:, atac.var.highly_variable].copy()
atac_hvf

In [ ]:
atac.write('../7_horizontal_RNA_ATAC/processed/atac.h5ad', compression='gzip')
atac_hvf.write('../7_horizontal_RNA_ATAC/processed/atac_hvf.h5ad', compression='gzip')

### norm

In [ ]:
atac.layers['counts'] = atac.X.copy()

In [ ]:
sc.pp.normalize_total(atac, target_sum=1e4)
sc.pp.log1p(atac)

In [ ]:
sc.pp.highly_variable_genes(atac, n_top_genes=60000, batch_key='sample')
atac

In [ ]:
atac_hvg = atac[:, atac.var.highly_variable].copy()
atac_hvg

In [ ]:
atac.write('../7_horizontal_RNA_ATAC/processed/atac_norm.h5ad', compression='gzip')
atac_hvg.write('../7_horizontal_RNA_ATAC/processed/atac_norm_hvg.h5ad', compression='gzip')

## MultiVI

In [ ]:
rna = sc.read_h5ad('../7_horizontal_RNA_ATAC/processed/rna_hvg.h5ad')
rna

In [ ]:
atac = sc.read_h5ad('../7_horizontal_RNA_ATAC/processed/atac_norm_hvg.h5ad')
atac

In [ ]:
import anndata as ad

In [ ]:
merged_adata = ad.concat([rna, atac], axis=1) 
merged_adata

In [ ]:
merged_adata.obs = rna.obs

In [ ]:
merged_adata

In [ ]:
merged_adata.var['modality'] = ['0'] * 4000 + ['1'] * 60000

In [ ]:
merged_adata

In [ ]:
merged_adata.write('../7_horizontal_RNA_ATAC/processed/rna_atac_norm_merge.h5ad', compression='gzip')

## UINMF

In [ ]:
data = sc.read_h5ad('../7_horizontal_RNA_ATAC/processed/rna_atac_tfidf_merge.h5ad')
data

In [ ]:
norm_data = sc.read_h5ad('../7_horizontal_RNA_ATAC/processed/rna_atac_norm_merge.h5ad')
norm_data

In [ ]:
data[:,4000:64000].X

In [ ]:
norm_data[:,4000:64000].X

In [ ]:
2079593942/140583274

In [ ]:
from scipy.io import mmwrite
from scipy.sparse import csr_matrix

# Assume sparse_matrix uses CSR format
mmwrite("../7_horizontal_RNA_ATAC/processed/uinmf/matrix_count.mtx", data.layers['counts'])

In [ ]:
cell = pd.DataFrame(data.obs_names)
cell

In [ ]:
cell.to_csv('../7_horizontal_RNA_ATAC/processed/uinmf/cell_norm.csv')

In [ ]:
gene = pd.DataFrame(data.var_names)
gene

In [ ]:
gene.to_csv('../7_horizontal_RNA_ATAC/processed/uinmf/feature_norm.csv')

# RunMultiVI

In [ ]:
from scipy.io import mmread
from scipy.sparse import csr_matrix
import anndata as ad
import pandas as pd
import scvi
import numpy as np
import scanpy as sc
scvi.settings.seed = 420
import os, sys, time

In [ ]:
dataset_id = "Dataset1/"
data_path = "../dataset/Horizontal/RNA_ATAC/"+dataset_id ## path to raw data

for i in range(1,3):
    batch_num = str(i)
    path = data_path+"batch"+batch_num
    cell_names = pd.read_csv(path+'/RNA/barcodes.tsv', sep = '\t', header=None, index_col=None)
    cell_names.columns =  ['cell_ids']
    X = csr_matrix(mmread(path+'/RNA/matrix.mtx').T)
    gene_names = pd.read_csv(path+'/RNA/genes.tsv', sep = '\t',  header=None, index_col=None)
    gene_names.columns =  ['gene_ids']
    locals()['adata_RNA'+str(i)] = ad.AnnData(X, obs=pd.DataFrame(index=cell_names.cell_ids), var=pd.DataFrame(index = gene_names.gene_ids))
    locals()['adata_RNA'+str(i)].var_names_make_unique()

    # peak information
    cell_names = pd.read_csv(path+'/ATAC/barcodes.tsv', sep = '\t', header=None, index_col=None)
    cell_names.columns =  ['cell_ids']
    X = csr_matrix(mmread(path+'/ATAC/matrix.mtx').T)
    peak_name = pd.read_csv(path+'/ATAC/peaks.tsv',header=None,index_col=None)
    peak_name.columns = ['peak_ids']
    locals()['adata_ATAC'+str(i)]  = ad.AnnData(X, obs=pd.DataFrame(index=cell_names.cell_ids), var=pd.DataFrame(index = peak_name.peak_ids))

del X

In [ ]:
adata_RNA = ad.concat([adata_RNA1,adata_RNA2])
adata_RNA.obs['batch'] = ['batch1']*adata_RNA1.n_obs + ['batch2']*adata_RNA2.n_obs
sc.pp.highly_variable_genes(
    adata_RNA,
    batch_key="batch",
    flavor="seurat_v3",
    n_top_genes=4000,
    subset=True
)
adata_RNA1 = adata_RNA[adata_RNA.obs['batch'] == 'batch1'].copy()
adata_RNA2 = adata_RNA[adata_RNA.obs['batch'] == 'batch2'].copy()

for i in range(1,3):
    locals()['adata_paired'+str(i)] = ad.concat([eval('adata_RNA'+str(i)), eval('adata_ATAC'+str(i))], merge = "same",axis=1)
    eval('adata_paired'+str(i)).obs_keys
    modality = ['Gene Expression']*eval('adata_RNA'+str(i)).shape[1]+['Peaks']*eval('adata_ATAC'+str(i)).shape[1]
    locals()['adata_paired'+str(i)].var['modality']=modality

adata_mvi = scvi.data.organize_multiome_anndatas(adata_paired1, adata_paired2)
del adata_paired1, adata_paired2

adata_mvi = adata_mvi[:, adata_mvi.var["modality"].argsort()].copy()

In [ ]:
adata_mvi

In [ ]:
adata_mvi.X

## test

In [ ]:
import torch

In [ ]:
model = torch.load("../results/Dataset1/model_mtv.pth")

In [ ]:
model

In [ ]:
adata_mvi.obsm['latent'] = model.get_latent_representation()

In [ ]:
meta_1 = pd.read_csv('../dataset/Horizontal/RNA_ATAC/Dataset1/Dataset1_batch1_celltype.csv',
                    index_col=0)
meta_2 = pd.read_csv('../dataset/Horizontal/RNA_ATAC/Dataset1/Dataset1_batch2_celltype.csv',
                    index_col=0)
meta = pd.concat([meta_1, meta_2])
meta

In [ ]:
adata_mvi.obs['celltype'] = meta['celltype'].tolist()

In [ ]:
sc.pp.neighbors(adata_mvi, use_rep='latent')
sc.tl.umap(adata_mvi)
latent = adata_mvi.obsm['latent'].copy()

In [ ]:
adata_mvi

In [ ]:
sc.pl.umap(adata_mvi, color=['modality', 'celltype'])

In [ ]:
adata_mvi.write_h5ad('../results/Dataset1/adata_mtv.h5ad', compression='gzip')

In [ ]:
np.savetxt("../results/Dataset1/latent_mtv.csv", latent, delimiter=',')

## test_blood

In [ ]:
os.chdir('../')

In [ ]:
adata_mvi = sc.read_h5ad('../results/blood/adata_mtv.h5ad')
adata_mvi

In [ ]:
a = pd.DataFrame(adata_mvi.obs['cellType'].value_counts())

In [ ]:
a

In [ ]:
data = pd.read_csv('/path/to/hemelas_project/z_others/6_ann_visual/visual/sample_all.csv', index_col=0)
data['sample_'] = data.index.tolist()
cell = adata.obs
all = pd.merge(cell, data, left_on='sample', right_on='sample_')
all.index = adata.obs_names.tolist()

In [ ]:
adata_mvi.obs = all

In [ ]:
sc.pl.umap(adata_mvi, color='technology')

In [ ]:
sc.pl.umap(adata_mvi, color='dataset')

In [ ]:
sc.pl.umap(adata_mvi, color='level_1')

In [ ]:
sc.pl.umap(adata_mvi, color='level_2')

In [ ]:
sc.pl.umap(adata_mvi, color='level_3')

## marker gene

In [ ]:
sc.pl.umap(adata_mvi, color=['CD79A', 'MS4A1'])

In [ ]:
sc.pl.umap(adata_mvi, color=['SPINK2', 'CRHBP'])

In [ ]:
sc.pl.umap(adata_mvi, color=['CD8A', 'CD3E'])

# uinmf_test

In [ ]:
import scanpy as sc
import pandas as pd
import numpy as np

In [ ]:
import os
os.chdir('/path/to/hemelas_project/z_others/')

In [ ]:
data = sc.read_h5ad('6_ann_visual/processed/adata_rna_celltypist_cellhint.h5ad')
data

In [ ]:
data.obsm['X_umap']

## uinmf umap

In [ ]:
umap = pd.read_csv('7_horizontal_RNA_ATAC/results/blood/uinmf_umap.csv', index_col=0)
umap

In [ ]:
uma = umap.reindex(data.obs_names)
uma

In [ ]:
data.obsm['X_umap'] = uma.to_numpy()

In [ ]:
sc.pl.umap(data, color='dataset')

In [ ]:
sc.pl.umap(data, color='level_1')

In [ ]:
sc.pl.umap(data, color='level_2')

In [ ]:
sc.pl.umap(data, color='level_3')

## umap_k30

In [ ]:
umap = pd.read_csv('7_horizontal_RNA_ATAC/results/blood/uinmf_umap_k30.csv', index_col=0)
umap

In [ ]:
uma = umap.reindex(data.obs_names)
uma

In [ ]:
data.obsm['X_umap'] = uma.to_numpy()

In [ ]:
sample = pd.read_csv('/path/to/hemelas_project/z_others/6_ann_visual/visual/sample_all.csv', index_col=0)
sample['sample_'] = sample.index.tolist()
cell = data.obs
all = pd.merge(cell, sample, left_on='sample', right_on='sample_')
all.index = data.obs_names.tolist()

In [ ]:
data.obs = all

In [ ]:
sc.pl.umap(data, color='technology')

In [ ]:
sc.pl.umap(data, color='level_1')

In [ ]:
sc.pl.umap(data, color='level_2')

In [ ]:
sc.pl.umap(data, color='level_3')